<h1 style="text-align:center; color:#00F1FA;">MCQ Creator App</h1>
<p style="text-align:center; color:#00F1FA;">Loads PDF documents, generates embeddings, stores them in Pinecone, retrieves relevant content, and structures the output as multiple choice questions.</p>

<h2 style="color:#4BFA00;">Table of Contents</h2>
<ul style="color:#4BFA00;">
    <li>Install and Import Dependencies</li>
    <li>Load Documents</li>
    <li>Transform Documents</li>
    <li>Generate Text Embeddings</li>
    <li>Vector Store - Pinecone</li>
    <li>Retrieve Answers</li>
    <li>Structure the Output</li>
</ul>

<h2 style="color:#FA00EE;">Install Libraries</h2>
<p style="color:#FA00EE;">Uncomment and run the lines below if any package is not installed in your environment.</p>

In [1]:
# Uncomment and run if packages are not installed
#!pip install openai==1.14.2
#!pip install langchain==0.1.13
#!pip install unstructured==0.12.3
#!pip install tiktoken==0.5.2
#!pip install pinecone-client==3.2.0
#!pip install pypdf==4.1.0
#!pip install sentence-transformers==2.5.1

<h2 style="color:#FA00EE;">Import Dependencies</h2>
<p style="color:#FA00EE;">Importing all required libraries for document loading, text splitting, embeddings, vector storage, and question answering.</p>

In [2]:
import os
import re
import json
import openai

# Document loading and splitting
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# Vector store
from langchain_community.vectorstores import Pinecone

# LLM
from langchain_openai import OpenAI

# QA chain
from langchain_classic.chains.question_answering import load_qa_chain

# Output parsing
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3

In [3]:
from dotenv import load_dotenv

# Load keys from .env file
load_dotenv(dotenv_path="/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/Multiple_Choice_Question_Creator_App/.env")

# Verify keys loaded
print("OPENAI KEY FOUND:", bool(os.getenv("OPENAI_API_KEY")))
print("PINECONE KEY FOUND:", bool(os.getenv("PINECONE_API_KEY")))
print("HUGGINGFACE KEY FOUND:", bool(os.getenv("HUGGINGFACEHUB_API_TOKEN")))

OPENAI KEY FOUND: True
PINECONE KEY FOUND: False
HUGGINGFACE KEY FOUND: False


<h2 style="color:#00F1FA;">Load Documents</h2>
<p style="color:#00F1FA;">Reads all PDF files from the Docs directory using PyPDFDirectoryLoader. Each PDF page is loaded as a separate document object.</p>

In [6]:
!pip install pypdf

  Using cached pypdf-6.9.2-py3-none-any.whl.metadata (7.1 kB)
Using cached pypdf-6.9.2-py3-none-any.whl (333 kB)


In [7]:
# Function to load all PDFs from a directory
def load_docs(directory):
    loader = PyPDFDirectoryLoader(directory)
    documents = loader.load()
    return documents

# Updated path to the correct directory
directory = '/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/Multiple_Choice_Question_Creator_App/'
documents = load_docs(directory)

# Check total number of pages loaded across all PDFs
print(f"Total pages loaded: {len(documents)}")

Total pages loaded: 5


<h2 style="color:#00F1FA;">Transform Documents</h2>
<p style="color:#00F1FA;">Splits the loaded documents into smaller chunks using RecursiveCharacterTextSplitter. This is necessary because LLMs have token limits and cannot process entire documents at once.</p>
<ul style="color:#00F1FA;">
    <li>chunk_size=1000 — each chunk contains up to 1000 characters</li>
    <li>chunk_overlap=20 — 20 characters overlap between consecutive chunks to preserve context at boundaries</li>
</ul>

In [8]:
# Function to split documents into chunks
def split_docs(documents, chunk_size=1000, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    docs = text_splitter.split_documents(documents)
    return docs

docs = split_docs(documents)

# Check total number of chunks created
print(f"Total chunks created: {len(docs)}")

Total chunks created: 13
